## Dataset

In [ ]:
import os
import torch
import random

import numpy as np
import pandas as pd
import scipy.sparse as sp

from pathlib import Path
from typing import Optional, Dict, Any

### Dataset Check

#### NBA Dataset

- 그래프 기반 fairness 연구용으로 가공된 social/attribute graph
    - Twitter social network +Kaggle NBA player 데이터 결합

- 선수들 간 관계 + 민감 속성 + 분류 라벨
    - 그래프 구조: Twitter 상에서의 관계
    - 노드 속성: Kaggle 선수 통계

- Nodes: 403명 NBA 선수
- Edges: 16,570개 / Twitter에서의 팔로우-관계 기반 연결 **농구 실력 관계가 아니라 사회적 네트워크 기반 그래프 (fairness 연구에서 일부러 이런 noisy graph를 사용함)
- Node Features: 95차원 / 경기 통계, 나이, 국적, 연봉 등 성능+인구통계 섞여 있어 fairness 연구에 적합
- Label: salary

In [ ]:
nba_df = pd.read_csv('./dataset/NBA/nba.csv')
nba_df.head()

,user_id,SALARY,AGE,MP,FG,FGA,FG%,3P,3PA,3P%,...,ORL/TOR,PHI,PHI/OKC,PHX,POR,SA,SAC,TOR,UTAH,WSH
0,105305397,-1,25,14.8,2.7,5.6,0.487,0.8,2.1,0.374,...,0,0,0,0,0,0,0,0,0,0
1,49680175,-1,32,4.9,0.9,2.4,0.383,0.6,1.4,0.400,...,0,0,0,0,0,0,0,0,0,0
2,364013199,0,20,20.1,2.4,5.9,0.399,0.6,2.0,0.321,...,0,0,0,0,0,0,0,0,0,0
3,234811698,0,25,14.0,2.1,4.8,0.440,0.5,1.4,0.337,...,0,0,0,0,0,0,0,0,0,0
4,1031967637561954304,1,28,8.4,2.1,3.8,0.545,0.0,0.0,0.000,...,0,0,0,0,0,0,0,0,0,0


In [10]:
nba_df.columns

Index(['user_id', 'SALARY', 'AGE', 'MP', 'FG', 'FGA', 'FG%', '3P', '3PA',
       '3P%', '2P', '2PA', '2P%', 'eFG%', 'FT', 'FTA', 'FT%', 'ORB', 'DRB',
       'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF_x', 'POINTS', 'GP', 'MPG',
       'ORPM', 'DRPM', 'RPM', 'WINS_RPM', 'PIE', 'PACE', 'W', 'player_height',
       'player_weight', 'country', 'C', 'PF_y', 'PF-C', 'PG', 'SF', 'SG',
       'ATL', 'ATL/CLE', 'ATL/LAL', 'BKN', 'BKN/WSH', 'BOS', 'CHA', 'CHI',
       'CHI/OKC', 'CLE', 'CLE/DAL', 'CLE/MIA', 'DAL', 'DAL/BKN', 'DAL/PHI',
       'DEN', 'DEN/CHA', 'DEN/POR', 'DET', 'GS', 'GS/CHA', 'GS/SAC', 'HOU',
       'HOU/LAL', 'HOU/MEM', 'IND', 'LAC', 'LAL', 'MEM', 'MIA', 'MIL',
       'MIL/CHA', 'MIN', 'NO', 'NO/DAL', 'NO/MEM', 'NO/MIL', 'NO/MIN/SAC',
       'NO/ORL', 'NO/SAC', 'NY', 'NY/PHI', 'OKC', 'ORL', 'ORL/TOR', 'PHI',
       'PHI/OKC', 'PHX', 'POR', 'SA', 'SAC', 'TOR', 'UTAH', 'WSH'],
      dtype='object')

In [15]:
nba_df['SALARY'].value_counts()

SALARY
 1    159
 0    154
-1     90
Name: count, dtype: int64

In [ ]:
sens_attr = "country"
pred_attr = "SALARY"
label_number = 100
sens_number = 50
seed = 20
test_idx = True

#### Pokec Dataset

- 슬로바키아 SNS "Pokec"에서 수집된 대규모 social network
- GNN fairness 논문에서 가장 자주 쓰이는 데이터 중 하나

- 그래프 구조
    - 노드: 사용자 user
    - 엣지: 친구 관계 friendship

- 규모
    - Nodes: 66~67k
    - Edges: 700~880k

- Feature: 260~270차원 / 나이, 성별, 관심사, 프로필 정보 (noisy + sparse + 현실적인 feature)
- Label: 직업/직종 work field
- Sensitive attribute: 지역 region (같은 지역끼리 더 많이 연결되는 homophily 존재)

- Pokec-z vs Pokec-n: 서로 다른 지역 subset, 그래프 구조와 분포가 약간 다름, generalization 확인을 위해 둘 다 사용

- 핵심 특징
    - 강한 homophily: 비슷한 사람끼리 연결됨, fairness 문제 더 심각해짐
    - 대규모 그래프: gnn의 진짜 성능 테스트 가능
    - 현실적인 bias 존재: 지역 기반 편향이 실제로 존재

In [11]:
pokec_df = pd.read_csv('./dataset/pokec/region_job.csv')
pokec_df.head()

,user_id,public,completion_percentage,gender,region,AGE,I_am_working_in_field,spoken_languages_indicator,anglicky,nemecky,...,odbornu literaturu,psychologicku literaturu,literaturu pre rozvoj osobnosti,cestopisy,literaturu faktu,poeziu,zivotopisne a pamate,pocitacovu literaturu,filozoficku literaturu,literaturu o umeni a architekture
0,1,1,14,1.0,0,26.0,-1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
1,131075,0,33,1.0,1,25.0,-1,1,0,1,...,0,0,0,0,0,0,0,0,0,0
2,5,1,66,1.0,0,26.0,3,1,1,1,...,0,0,0,0,0,0,0,0,0,0
3,6,0,22,0.0,0,38.0,-1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,7,1,12,0.0,1,22.0,-1,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [12]:
pokec_df.columns

Index(['user_id', 'public', 'completion_percentage', 'gender', 'region', 'AGE',
       'I_am_working_in_field', 'spoken_languages_indicator', 'anglicky',
       'nemecky',
       ...
       'odbornu literaturu', 'psychologicku literaturu',
       'literaturu pre rozvoj osobnosti', 'cestopisy', 'literaturu faktu',
       'poeziu', 'zivotopisne a pamate', 'pocitacovu literaturu',
       'filozoficku literaturu', 'literaturu o umeni a architekture'],
      dtype='object', length=279)

In [14]:
pokec_df['I_am_working_in_field'].value_counts()

I_am_working_in_field
-1    57534
 0     4764
 1     1964
 3     1287
 2     1266
 4      981
Name: count, dtype: int64

In [ ]:
sens_attr = "region"
pred_attr = "I_am_working_in_field"
label_number = 500  # the number of labels
sens_number = 200   # the number of sensitive attributes
seed = 20
test_idx = False

#### German Dataset

- NiFTY에서 만든 금융 그래프
- German Credit dataset으로 금융 대출 데이터
- NiFTY 논문에서 이걸 graph로 인위적으로 변환함

- 그래프 생성 방식
    - 노드: 사람 bank customer
    - 엣지: feature similarity 기반 연결 / income, credit history 비슷하면 연결 -> synthetic graph

- 규모 (굉장히 작음)
    - nodes: 1,000
    - edges: 34,970
    - feature: 27

- Features: 금융 정보 / 신용기록, 대출상태, 소득 등
- Label: 대출 승인 여부 credit risk
- Sensitive attribute: gender 또는 age


In [17]:
german_df = pd.read_csv('./dataset/NIFTY/german.csv')
german_df.head()

,GoodCustomer,Gender,ForeignWorker,Single,Age,LoanDuration,PurposeOfLoan,LoanAmount,LoanRateAsPercentOfIncome,YearsAtCurrentHome,...,OtherLoansAtBank,OtherLoansAtStore,HasCoapplicant,HasGuarantor,OwnsHouse,RentsHouse,Unemployed,YearsAtCurrentJob_lt_1,YearsAtCurrentJob_geq_4,JobClassIsSkilled
0,1,Male,0,1,67,6,Electronics,1169,4,4,...,0,0,0,0,1,0,0,0,1,1
1,-1,Female,0,0,22,48,Electronics,5951,2,2,...,0,0,0,0,1,0,0,0,0,1
2,1,Male,0,1,49,12,Education,2096,2,3,...,0,0,0,0,1,0,0,0,1,0
3,1,Male,0,1,45,42,Furniture,7882,2,4,...,0,0,0,1,0,0,0,0,1,1
4,-1,Male,0,1,53,24,NewCar,4870,3,4,...,0,0,0,0,0,0,0,0,0,1


In [19]:
german_df.columns

Index(['GoodCustomer', 'Gender', 'ForeignWorker', 'Single', 'Age',
       'LoanDuration', 'PurposeOfLoan', 'LoanAmount',
       'LoanRateAsPercentOfIncome', 'YearsAtCurrentHome',
       'NumberOfOtherLoansAtBank', 'NumberOfLiableIndividuals', 'HasTelephone',
       'CheckingAccountBalance_geq_0', 'CheckingAccountBalance_geq_200',
       'SavingsAccountBalance_geq_100', 'SavingsAccountBalance_geq_500',
       'MissedPayments', 'NoCurrentLoan', 'CriticalAccountOrLoansElsewhere',
       'OtherLoansAtBank', 'OtherLoansAtStore', 'HasCoapplicant',
       'HasGuarantor', 'OwnsHouse', 'RentsHouse', 'Unemployed',
       'YearsAtCurrentJob_lt_1', 'YearsAtCurrentJob_geq_4',
       'JobClassIsSkilled'],
      dtype='object')

In [ ]:
pokec_df['GoodCustomer'].value_counts() # 1:good

GoodCustomer
 1    700
-1    300
Name: count, dtype: int64

In [ ]:
sens_attr = "GoodCustomer" # or "Age"
pred_attr = "I_am_working_in_field"
label_number = 500  # the number of labels
sens_number = 200   # the number of sensitive attributes
seed = 20
test_idx = False

### Dataset Load

In [2]:
# dataset configs
DATASET_CONFIGS = {
    "nba": {
        "csv_file": "nba.csv",
        "edge_file": "nba_relationship.txt",
        "id_col": "user_id",
        "default_sens_attr": "country",
        "default_predict_attr": "SALARY",
    },

    "pokec_z": {
        "csv_file": "region_job.csv",  
        "edge_file": "region_job_relationship.txt",
        "id_col": "user_id",
        "default_sens_attr": "region",
        "default_predict_attr": "I_am_working_in_field",
    },

    "pokec_n": {
        "csv_file": "region_job_2.csv",  
        "edge_file": "region_job_2_relationship.txt",
        "id_col": "user_id",
        "default_sens_attr": "region",
        "default_predict_attr": "I_am_working_in_field",
    },

    "german": {
        "csv_file": "german.csv",
        "edge_file": "german_edges.txt",  
        "id_col": None,
        "default_sens_attr": "Gender",
        "default_predict_attr": "GoodCustomer",
    },
}


In [3]:
# dataset load
def _build_symmetric_adj(num_nodes: int, edges: np.ndarray) -> sp.coo_matrix:
    adj = sp.coo_matrix(
        (np.ones(edges.shape[0]), (edges[:, 0], edges[:, 1])),
        shape=(num_nodes, num_nodes),
        dtype=np.float32,
    )
    adj = adj + adj.T.multiply(adj.T > adj) - adj.multiply(adj.T > adj)
    adj = adj + sp.eye(adj.shape[0], dtype=np.float32)  # self-loop
    return adj.tocoo()

def _load_edges(edge_path: Path, node_ids: np.ndarray, id_col: Optional[str]) -> np.ndarray:
    edges_unordered = np.genfromtxt(edge_path, dtype=int)
    if edges_unordered.ndim == 1:
        edges_unordered = edges_unordered.reshape(-1, 2)

    if id_col is None:
        # edge file가 이미 0 ~ N-1 index 기준이라고 가정
        edges = edges_unordered
    else:
        idx_map = {node_id: i for i, node_id in enumerate(node_ids)}
        mapped = list(map(idx_map.get, edges_unordered.flatten()))
        if any(v is None for v in mapped):
            raise ValueError(f"edge file {edge_path} 안에 CSV의 id와 매칭되지 않는 값이 있습니다.")
        edges = np.array(mapped, dtype=int).reshape(edges_unordered.shape)

    return edges

def _encode_sensitive_column(series: pd.Series) -> np.ndarray:
    """
    민감 속성을 0/1 또는 정수형으로 변환.
    문자열이면 category code로 바꾼다.
    """
    if pd.api.types.is_numeric_dtype(series):
        return series.values

    # 문자열/범주형 처리
    cat = pd.Categorical(series)
    return cat.codes.astype(np.float32)

def load_fair_graph_dataset(
    dataset_name: str,
    root: str,
    sens_attr: Optional[str] = None,
    predict_attr: Optional[str] = None,
    label_number: Optional[int] = 1000,
    sens_number: Optional[int] = 500,
    use_all_sensitive: bool = False,
    sens_ratio: Optional[float] = None,
    train_ratio: float = 0.5,
    val_ratio: float = 0.25,
    seed: int = 19,
    test_idx_as_val: bool = False,
    feature_drop_cols: Optional[list] = None,
) -> Dict[str, Any]:
    if dataset_name not in DATASET_CONFIGS:
        raise ValueError(f"지원하지 않는 dataset_name: {dataset_name}")

    cfg = DATASET_CONFIGS[dataset_name]
    root = Path(root)

    csv_path = root / cfg["csv_file"]
    edge_path = root / cfg["edge_file"] if cfg.get("edge_file") else None

    sens_attr = sens_attr or cfg["default_sens_attr"]
    predict_attr = predict_attr or cfg["default_predict_attr"]
    id_col = cfg["id_col"]

    df = pd.read_csv(csv_path)

    if sens_attr not in df.columns:
        raise ValueError(f"sens_attr '{sens_attr}'가 없습니다.")
    if predict_attr not in df.columns:
        raise ValueError(f"predict_attr '{predict_attr}'가 없습니다.")
    if id_col is not None and id_col not in df.columns:
        raise ValueError(f"id_col '{id_col}'가 없습니다.")

    # 1) drop 대상 정리
    drop_cols = [sens_attr, predict_attr]
    if id_col is not None:
        drop_cols.append(id_col)
    if feature_drop_cols:
        drop_cols.extend(feature_drop_cols)

    feature_cols = [c for c in df.columns if c not in set(drop_cols)]

    # 2) feature one-hot encoding
    feature_df = df[feature_cols].copy()
    feature_df = pd.get_dummies(feature_df, drop_first=False)

    features = sp.csr_matrix(feature_df.values, dtype=np.float32)

    # 3) label 처리
    labels = df[predict_attr].values
    labels = np.asarray(labels)

    # 4) sensitive 처리
    sens = _encode_sensitive_column(df[sens_attr])

    # 5) graph 처리
    if id_col is not None:
        node_ids = np.array(df[id_col], dtype=int)
    else:
        node_ids = np.arange(len(df), dtype=int)

    if edge_path is not None and edge_path.exists():
        edges = _load_edges(edge_path, node_ids=node_ids, id_col=id_col)
        adj = _build_symmetric_adj(num_nodes=len(df), edges=edges)
    else:
        adj = sp.eye(len(df), dtype=np.float32).tocoo()

    # 6) tensor 변환
    features = torch.FloatTensor(np.asarray(features.todense()))
    labels = torch.LongTensor(labels)
    sens = torch.FloatTensor(sens)

    # 7) semi-supervised split
    rng = random.Random(seed)
    label_idx = np.where(labels.numpy() >= 0)[0].tolist()
    rng.shuffle(label_idx)

    n_total = len(label_idx)
    n_train_full = int(train_ratio * n_total)
    n_val = int(val_ratio * n_total)

    n_train = n_train_full if label_number is None else min(n_train_full, label_number)

    idx_train = label_idx[:n_train]
    idx_val = label_idx[n_train_full:n_train_full + n_val]

    if test_idx_as_val:
        idx_test = label_idx[n_train:]
        idx_val = idx_test
    else:
        idx_test = label_idx[n_train_full + n_val:]

    # 8) 민감 속성 공개 subset 선택
    sens_idx = set(np.where(sens.numpy() >= 0)[0].tolist())
    idx_test = np.asarray(list(sens_idx & set(idx_test)), dtype=int)

    candidate_sens_train = list(sens_idx - set(idx_val) - set(idx_test))
    rng.shuffle(candidate_sens_train)

    if use_all_sensitive:
        selected_sens_train = candidate_sens_train
    else:
        if sens_ratio is not None:
            k = int(len(candidate_sens_train) * sens_ratio)
        elif sens_number is not None:
            k = min(sens_number, len(candidate_sens_train))
        else:
            k = len(candidate_sens_train)
        selected_sens_train = candidate_sens_train[:k]

    return {
        "adj": adj,
        "features": features,
        "labels": labels,
        "sens": sens,
        "idx_train": torch.LongTensor(idx_train),
        "idx_val": torch.LongTensor(idx_val),
        "idx_test": torch.LongTensor(idx_test),
        "idx_sens_train": torch.LongTensor(selected_sens_train),
        "feature_names": list(feature_df.columns),
        "df": df,
    }

In [4]:
# pokec_z
pokec_z = load_fair_graph_dataset(
    dataset_name="pokec_z",
    root="./dataset/pokec/",
    sens_attr="region",
    predict_attr="I_am_working_in_field",
    label_number=500,
    sens_ratio=0.2,
    use_all_sensitive=False,
    test_idx_as_val=False, 
    seed=11,
)

# nba
nba = load_fair_graph_dataset(
    dataset_name="nba",
    root="./dataset/NBA/",
    sens_attr="country",
    predict_attr="SALARY",
    label_number=100,
    sens_ratio=0.2,
    use_all_sensitive=False,
    test_idx_as_val=False, 
    seed=11,
)


## Analysis

- 그래프에서 편향은 균일하지 않음: Pokec_z에서는 공정성 위험이 그래프 전반에 균일하게 분포하지 않고, 특정 구조적 위치에 집중된다는 것을 밝히는 분석
- pokec_z에서 공정성 위험은 그래프 전체에 균일하게 퍼져 있지 않고, minority isolation, boundary exposure, influence, structural instability와 같은 구조적 요인에 따라 특정 노드 및 neighborhood에 집중된다.
    - 따라서 uniform fairness intervention보다는 구조적으로 취약한 위치에 선택적으로 fairness budget을 배분하는 접근이 더 타당하다.

In [7]:
import os
import copy
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import GCNConv
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

def set_seed(seed: int = 11):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
# device
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda:1


In [ ]:
# 2-layer GCN
class BasicGCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, dropout=0.5):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, 1)
        self.dropout = dropout

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        h = self.conv1(x, edge_index)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)

        h = self.conv2(h, edge_index)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)

        logits = self.classifier(h).squeeze(-1)   # [N]
        return logits, h

## Modeling

- 기존 방법은 대체로 “representation을 공정하게 만들자” 혹은 “민감 속성을 제거하자”라는 전역적 관점이 강하다.

- 반면 제안 방법은 공정성 개입의 위치와 강도를 구조적으로 선택한다는 점이 다르다.
- 즉, 차별점은 단순히 debiasing objective가 아니라 intervention allocation mechanism에 있다.
- 또한 uncertainty를 쓰더라도 예측 confidence 자체를 말하는 게 아니라 구조 교란에 대한 불안정성을 fairness intervention과 연결한다는 점이 차별적이다.

- 문제 정의: 그래프에서 fairness violation은 구조적으로 불균일하게 발생하며, uniform fairness intervention은 비효율적일 수 있다.
- 핵심 가설: 구조적으로 fairness risk가 높은 노드에 fairness intervention을 선택적으로 배분하면, 동일한 global intervention보다 더 나은 fairness-utility trade-off를 얻을 수 있다.
- 핵심 메커니즘: 정적 구조 위험도 + 동적 structural uncertainty → node-level priority score → weighted/selective fairness intervention

- 방법론적 기여: 
    - 구조 기반 fairness risk 정식화
    - 학습 중 structural uncertainty 추정
    - priority-aware fairness optimization

- 실험적 기여:
    - global fairness 개선
    - local/neighborhood fairness 개선
    - utility 손실 완화
    - risk score의 해석 가능성 검증

- 그래프의 모든 노드/엣지/표현에 균일한 공정성 제약을 가하는 대신, 구조적으로 공정성에 취약한 위치에 선택적으로 fairness intervention을 배분하는 multi-level Fair GNN framework
    - bias는 local하고 구조적인데, intervention은 global하고 unifrom한 불일치가 기존 방법의 한계
- 그래프 전체에 균일한 공정성 제약을 가하는 대신, 구조적으로 공정성에 취약한 노드를 식별하고, 그 우선순위에 따라 multi-level fairness intervention을 선택적으로 배분하는 프레임워크를 제안
- 제안하는 Fair GNN 프레임워크는 공정성 제약을 전역적으로 균일하게 부과하는 대신, FIPS를 통해 구조적으로 취약한 노드를 식별하고, 그 우선순위에 따라 edge-level, representation-level, output-level의 fairness intervention을 선택적으로 조절하는 multi-level selective fairness framework

- 기존 Fair GNN 연구 한계 -> uniform intervention: 그래프 전반에 비슷한 강동의 fairness pressure
    - 전역 representation debiasing: 전체 임베딩에서 민감 속성 정보를 없애거나 그룹 간 분포를 맞추는 방식
    - 구조 자체를 전역적으로 수정하는 방식: 특정 edge를 drop하거나 reweight하는 방식

- 그래프에서 편향은 균일하지 않음
    - minority가 고립된 neighborhood
    - cross-group exposure가 높은 boundary region
    - bridge/hub처럼 편향 전파력이 큰 노드
    - 구조 교란에 민감한 불안정한 노드

- 핵심 아이디어: FIPS-guided multi-level selective fairness
    - 공정성 개입을 multi-level로 수행하되, 그 강도와 위치를 FIPS로 선택적으로 제아한다.
    - multi-level
        - 그래프 편향은 한 레벨에서만 생기지 않음
        - 한 군데만 개입하면 부족할 수 있지만, 모든 레벨에 무차별적으로 개입하면 과함
        - message/edge level: 어떤 이웃 정보를 얼마나 섞느냐가 문제일 수 있음
        - representation/neighborhood level: 그룹 정보가 latent space에 얽히는 게 문제일 수 있음
        - prediction/output level: 최종 분포가 집단별로 다르게 나타나는 게 문제일 수 있음
    - FIPS는 각 노드에서 계산되는 fairness intervention priority score -> fairness controller의 입력
        - 이 노드가 불공정하다 X
        - 구조적으로 fairness risk가 높고
        - 학습 중 구조적으로 불안정하며
        - 편향 전파 관점에서 중요할 가능성이 큰 노드일수록 더 높은 값을 가짐

- 전체 프레임워크 (4개 모듈)
    - 1. Graph Encoder: 기본 gnn encoder
    - 2. FIPS Estimator: 각 노드의 fairness intervention priority score 계산
    - 3. Multi-level Fairness Intervention Module: FIPS 바탕으로 3 레벨 intervention 수행
    - 4. Joint Optimization: task loss와 selective fairness losses를 함께 최적화

Step 1
정적 구조 fairness risk 정의
예: self-group exposure deficit + local group mixture + structural influence

Step 2
구조 perturbation 기반 dynamic uncertainty 측정
예: edge dropout 여러 번 수행 후 embedding/prediction variance 측정

Step 3
둘을 결합해 node-level priority score 계산

Step 4
이 점수로 node-wise fairness loss weight 조정

Step 5
global fairness + local fairness + utility + calibration/interpretability 평가